# Reproduce the TurboQuant halfword decode regression

Run all cells from any directory in an LMCache checkout. The notebook first executes all four direct CUDA serializer round trips, then compiles and benchmarks the fixed decode kernel with a Llama-8B-style 8K-token shape. Any failed or skipped regression stops the notebook.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
from pathlib import Path
import os
import subprocess
import sys

import torch

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "lmcache").is_dir()
)
TEST_FILE = ROOT / "tests/v1/distributed/serde/test_turboquant.py"
BENCHMARK = ROOT / "examples/serde/turboquant/bench_turboquant.py"
assert TEST_FILE.is_file() and BENCHMARK.is_file()
assert torch.cuda.is_available(), "CUDA is required"
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(ROOT) + os.pathsep + ENV.get("PYTHONPATH", "")
print(
    {
        "root": str(ROOT),
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
    }
)

In [ ]:
# SPDX-License-Identifier: Apache-2.0
test_command = [
    sys.executable,
    "-m",
    "pytest",
    "-q",
    "-s",
    str(TEST_FILE),
    "-k",
    "test_turboquant_direct_roundtrip_cuda",
]
print("Running:", " ".join(test_command))
tests = subprocess.run(
    test_command,
    cwd=ROOT,
    env=ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(tests.stdout)
tests.check_returncode()
assert "4 passed" in tests.stdout, "expected all four CUDA presets to run"

In [ ]:
# SPDX-License-Identifier: Apache-2.0
benchmark_command = [
    sys.executable,
    str(BENCHMARK),
    "--device",
    "cuda",
    "--dtype",
    "bfloat16",
    "--layers",
    "32",
    "--blocks",
    "512",
    "--block-size",
    "16",
    "--kv-heads",
    "8",
    "--head-dim",
    "128",
    "--warmup",
    os.getenv("LMCACHE_TQ_WARMUP", "3"),
    "--iters",
    os.getenv("LMCACHE_TQ_ITERS", "10"),
]
print("Running:", " ".join(benchmark_command))
benchmark = subprocess.run(
    benchmark_command,
    cwd=ROOT,
    env=ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(benchmark.stdout)
benchmark.check_returncode()
assert "2x32x8192x1024" in benchmark.stdout
for preset in (
    "turboquant_k8v4",
    "turboquant_4bit_nc",
    "turboquant_k3v4_nc",
    "turboquant_3bit_nc",
):
    assert preset in benchmark.stdout, preset
print({"status": "passed", "shape": "2x32x8192x1024", "presets": 4})